# SAR-to-EO Image Translation — GalaxEye Assignment
**Model:** Pix2Pix U-Net + PatchGAN  
**Loss:** L1 + Adversarial + FFT Frequency + VGG Perceptual  
**Dataset:** Sentinel-1&2 terrain-split (Kaggle)


In [ ]:
# CELL 1: Check GPU
import subprocess, os, torch
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 2: Install required packages
!pip install lpips pytorch-fid scikit-image -q

In [ ]:
# CELL 3: Clone repo
import os, subprocess
subprocess.run(
    ['git', 'clone', 'https://github.com/Trafalgar-2006/sar2eo.git', '/kaggle/working/sar2eo'],
    check=True
)
os.chdir('/kaggle/working/sar2eo')
os.makedirs('outputs', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('logs', exist_ok=True)
print('Working directory:', os.getcwd())
print('Files:', sorted(os.listdir('.')))

In [ ]:
# CELL 4: Verify dataset is attached
import os

# Check both common Kaggle input paths
possible_paths = [
    '/kaggle/input/sentinel12-image-pairs-segregated-by-terrain',
    '/kaggle/input/sentinel-1-2-image-pairs-segregated-by-terrain',
]

DATASET_PATH = None
for p in possible_paths:
    if os.path.exists(p):
        DATASET_PATH = p
        break

if DATASET_PATH is None:
    # List whatever is in /kaggle/input so we can see the correct name
    available = os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else []
    print('ERROR: Dataset not found!')
    print('Available inputs:', available)
    print('Please add the dataset via the + Add Input button and restart.')
else:
    print(f'Dataset found at: {DATASET_PATH}')
    for item in sorted(os.listdir(DATASET_PATH)):
        item_path = os.path.join(DATASET_PATH, item)
        if os.path.isdir(item_path):
            print(f'  {item}/ ({len(os.listdir(item_path))} items)')

In [ ]:
# CELL 5: Update config with CORRECT dataset path
import yaml, os

# Use the DATASET_PATH found in Cell 4
# If Cell 4 failed, set it manually here:
# DATASET_PATH = '/kaggle/input/YOUR-ACTUAL-PATH'

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['dataset_type'] = 'kaggle'
cfg['data']['kaggle_root']  = DATASET_PATH
cfg['data']['subset_size']  = None
cfg['training']['batch_size'] = 4
cfg['training']['epochs']     = 100
cfg['training']['num_workers'] = 2

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Config updated:')
print(f"  kaggle_root  : {cfg['data']['kaggle_root']}")
print(f"  batch_size   : {cfg['training']['batch_size']}")
print(f"  epochs       : {cfg['training']['epochs']}")

In [ ]:
# CELL 6: Smoke test (2 epochs, 100 samples) - MUST PASS before full training
import yaml, subprocess, copy

with open('config.yaml') as f:
    cfg_orig = yaml.safe_load(f)

cfg_smoke = copy.deepcopy(cfg_orig)
cfg_smoke['data']['subset_size']   = 100
cfg_smoke['training']['epochs']    = 2
cfg_smoke['training']['val_freq']  = 1
cfg_smoke['training']['save_freq'] = 2
cfg_smoke['active_ablation']       = 'full'

with open('config_smoke.yaml', 'w') as f:
    yaml.dump(cfg_smoke, f)

print('Running smoke test (2 epochs, 100 pairs)...')
result = subprocess.run(
    ['python', 'train.py', '--config', 'config_smoke.yaml', '--ablation', 'full'],
    capture_output=False
)
if result.returncode == 0:
    print('\nSmoke test PASSED -- safe to run full training')
else:
    print('\nSmoke test FAILED -- do NOT proceed to full training')

In [ ]:
# CELL 7a: Train Config A -- L1 only (baseline)
import subprocess
print('Training Config A: L1 only...')
subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_only'])
print('Config A done!')

In [ ]:
# CELL 7b: Train Config B -- L1 + Adversarial
import subprocess
print('Training Config B: L1 + Adversarial...')
subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_adv'])
print('Config B done!')

In [ ]:
# CELL 7c: Train Config C -- L1 + Adversarial + FFT
import subprocess
print('Training Config C: L1 + Adversarial + FFT...')
subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_adv_fft'])
print('Config C done!')

In [ ]:
# CELL 7d: Train Config D -- Full model (MAIN)
import subprocess
print('Training Config D (MAIN MODEL): Full loss stack...')
subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'full'])
print('Config D done!')

In [ ]:
# CELL 8: Evaluate all configs
import os, subprocess
for ablation in ['l1_only', 'l1_adv', 'l1_adv_fft', 'full']:
    ckpt = f'checkpoints/{ablation}/best.pth'
    if not os.path.exists(ckpt):
        print(f'Skipping {ablation} -- no checkpoint at {ckpt}')
        continue
    print(f'\n=== Evaluating: {ablation} ===')
    subprocess.run(['python', 'eval.py', '--config', 'config.yaml', '--weights', ckpt, '--split', 'test'])
print('\nAll evaluations complete!')

In [ ]:
# CELL 9: Zip and download results
import shutil, os
shutil.make_archive('/kaggle/working/sar2eo_results', 'zip', '.', 'outputs')
shutil.make_archive('/kaggle/working/sar2eo_checkpoints', 'zip', '.', 'checkpoints')
for f in ['/kaggle/working/sar2eo_results.zip', '/kaggle/working/sar2eo_checkpoints.zip']:
    size_mb = os.path.getsize(f) / 1e6 if os.path.exists(f) else 0
    print(f'  {f}  ({size_mb:.1f} MB)')
print('Download from the Kaggle output panel.')